# 01 — Build and verify kNN graphs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gokhanturan/apsp-knn-benchmark/blob/main/notebooks/01_build_graphs.ipynb)

This notebook reconstructs the graph inputs used in the manuscript. It verifies the 12 original graphs against the archived `.npz` files and can optionally rebuild the full controlled Digits scaling set.

**Important:** features are standardized within the corresponding dataset/subsample, Euclidean distance is used, and directed kNN neighborhoods are converted to an undirected union graph.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_NAME = "apsp-knn-benchmark"
REPO_URL = "https://github.com/gokhanturan/apsp-knn-benchmark.git"

# Colab: clone the repository if the notebook was opened directly from GitHub.
if Path('/content').exists() and not (Path('/content') / REPO_NAME).exists():
    subprocess.run(['git', 'clone', REPO_URL, str(Path('/content') / REPO_NAME)], check=True)

if (Path('/content') / REPO_NAME).exists():
    ROOT = Path('/content') / REPO_NAME
else:
    # Local/Jupyter execution from repo/notebooks or repo root.
    cwd = Path.cwd().resolve()
    ROOT = cwd.parent if cwd.name == 'notebooks' else cwd

os.chdir(ROOT)
print('Repository root:', ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)


In [ ]:
import sys
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import pandas as pd
from scipy.sparse import load_npz
from graph_utils import build_original_graphs, build_scaling_graphs

DATA_DIR = ROOT / 'data'
ARCHIVED_GRAPHS = ROOT / 'graphs'
OUT_DIR = ROOT / 'reproduced_graphs'
OUT_DIR.mkdir(exist_ok=True)


## Rebuild the 12 original graphs

In [ ]:
metadata = build_original_graphs(DATA_DIR, OUT_DIR)
metadata

## Verify reconstructed graphs against the archived manuscript inputs

In [ ]:
checks = []
for dataset in ['wine', 'diabetes', 'breast_cancer', 'digits']:
    for k in [5, 10, 20]:
        a = load_npz(OUT_DIR / f'{dataset}_k{k}.npz').tocsr()
        b = load_npz(ARCHIVED_GRAPHS / f'{dataset}_k{k}.npz').tocsr()
        diff = a - b
        max_abs = float(np.max(np.abs(diff.data))) if diff.nnz else 0.0
        checks.append({
            'dataset': dataset,
            'k': k,
            'same_shape': a.shape == b.shape,
            'same_nnz': a.nnz == b.nnz,
            'allclose': bool(np.allclose(a.toarray(), b.toarray(), rtol=1e-12, atol=1e-12)),
            'max_abs_diff': max_abs,
        })
checks = pd.DataFrame(checks)
checks

In [ ]:
assert checks['allclose'].all(), 'At least one reconstructed original graph differs from the archived graph.'
print('All 12 original graphs reproduced successfully.')

## Optional: rebuild the controlled Digits scaling graphs

The full scaling set contains many graph files. Set `RUN_FULL_SCALING = True` to recreate them.

In [ ]:
RUN_FULL_SCALING = False

if RUN_FULL_SCALING:
    scaling_out = ROOT / 'reproduced_scaling_graphs'
    scaling_meta = build_scaling_graphs(DATA_DIR / 'digits.csv', scaling_out)
    display(scaling_meta.head())
    print('Generated scaling graphs:', len(scaling_meta))
else:
    print('Skipped full scaling-graph rebuild. Set RUN_FULL_SCALING=True to run it.')

The archived `scaling_graphs/` directory is already included so that the benchmark and analysis notebooks can run without first rebuilding every controlled-scaling graph.